In [180]:
import numpy as np
import pandas as pd
import re
from spellchecker import SpellChecker
import json ,csv
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')




In [ ]:
df =  pd.read_csv("..\\data\\processed\\dataset.csv")

In [182]:
df.head()

,id,Contains,SideEffect,Chemical_Class,Habit_Forming,Therapeutic_Class
0,1,Haloperidol (0.5mg),Most side effects do not require any medical a...,Butyrophenone Derivative,No,NEURO CNS
1,2,Bevacizumab (100mg),Most side effects do not require any medical a...,Monoclonal antibody (mAb),No,ANTI NEOPLASTICS
2,3,Darbepoetin alfa (40mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED
3,4,Darbepoetin alfa (25mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED
4,5,Darbepoetin alfa (60mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED


In [183]:
df.isna().sum()

id                       0
Contains                 0
SideEffect               0
Chemical_Class       91334
Habit_Forming            0
Therapeutic_Class        0
dtype: int64

In [184]:
df.columns

Index(['id', 'Contains', 'SideEffect', 'Chemical_Class', 'Habit_Forming',
       'Therapeutic_Class'],
      dtype='object')

In [185]:
sef = df[["id","SideEffect"]]

In [186]:
sef["SideEffects"] = sef["SideEffect"].apply(
    lambda x: "##".join(line.strip() for line in str(x).splitlines() if line.strip())
)


In [187]:
sef.head()

,id,SideEffect,SideEffects
0,1,Most side effects do not require any medical a...,Most side effects do not require any medical a...
1,2,Most side effects do not require any medical a...,Most side effects do not require any medical a...
2,3,Most side effects do not require any medical a...,Most side effects do not require any medical a...
3,4,Most side effects do not require any medical a...,Most side effects do not require any medical a...
4,5,Most side effects do not require any medical a...,Most side effects do not require any medical a...


In [188]:
def shrink_first_two(text):
    parts = str(text).split("##")
    for i in range(min(2, len(parts))):
        words = parts[i].split()
        parts[i] = " ".join(words[:3])
    
    return "##".join(parts)

In [189]:
sef["SideEffects"] = sef["SideEffects"].apply(shrink_first_two)

In [190]:
def split_first_two_rest(text):
    parts = str(text).split("##")
    
    first = parts[0] if len(parts) > 0 else None
    second = parts[1] if len(parts) > 1 else None
    
    rest = "##".join(parts[2:]) if len(parts) > 2 else None
    return pd.Series([first, second, rest])

In [191]:
sef[["line1", "line2", "side"]] = sef["SideEffects"].apply(split_first_two_rest)


In [192]:
words = ['effects', 'symptoms', 'issues', "Suspinion", "Oral", "Drops",
         "Tablet", "Suspension", "DT", "Syrup", "Liquid", "does",
         "Raspberry", "Dry","Kid","Drop","PD","DS","LS"]

pattern = r'\b(' + '|'.join(map(re.escape, words)) + r')\b|\d+'

sef['line1'] = sef['line1'].replace(pattern, np.nan, regex=True)

In [193]:
sef["line1"].unique()

array([nan, '":null', 'Azibact LR Readymix', 'Azilide -XL Redimed',
       'Adcef Insta Use', 'Ampoxin LB Capsule', 'Amoxy Bill CV',
       'Anti CC Plus', 'Al Won Plus', 'Azipro Xl Rediuse',
       'Brufen P Junior', 'B G Cold', 'B Cillin CV', 'Brikmox CL BD',
       'BD Cef O', 'Bio Nim KID', 'Cefolac Powder for',
       'Cipla Kids Billargic', 'Clavam LB Bid', 'CT Cold Plus',
       'CEFI O TABLET', 'Cadi O M', 'Clymo Forte Powder', 'Codmox CV Duo',
       'Dolocold Junior Nasal', 'D Lit M', 'Dolomec MF Plus',
       'Easyfour-TT Paediatric Vaccine', 'EL Clav BD', 'DD Norm Sachet',
       'Flamox CV Dds', 'Flublast Junior Nasal', 'Fealmox CV Forte',
       'Flox PC OR', 'Figomox CV Duo', 'Genomox CV Forte',
       'Genmox CV Forte', 'Grandmox Bid O', 'Hepatitis B Vaccine',
       'His Block M', 'H R T', 'Hipen Clav Dds', 'Helomox CV Forte',
       'Itmol M Forte', 'Ingee P Nasal', 'Ixoma CV Duo', 'I Tax OD',
       'Imox Clo LB', 'I Tax OF', 'I Cefod CV', 'JV Flox SM', 'J P Nimu',
 

In [194]:
sef["line2"].unique()

array(['Common side effects', None], dtype=object)

In [195]:
sef.drop(["line1","line2"],axis=1,inplace=True)
sef.head()

,id,SideEffect,SideEffects,side
0,1,Most side effects do not require any medical a...,Most side effects##Common side effects##Agitat...,Agitation##Extrapyramidal symptoms##Insomnia (...
1,2,Most side effects do not require any medical a...,Most side effects##Common side effects##Rectal...,Rectal bleeding##Taste change##Headache##Noseb...
2,3,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...
3,4,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...
4,5,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...


In [196]:
def normalize(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text).lower()
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [197]:
spell = SpellChecker()
def correct_spelling(text):
    corrected = []
    for word in text.split():
        corrected.append(spell.correction(word) or word)
    return " ".join(corrected)

In [198]:
all_unique_values = set()

for cell in sef['side'].dropna():
    if str(cell).strip() != "":
        terms = [t.strip() for t in str(cell).split('##') if t.strip()]
        all_unique_values.update(terms)

print(f"unique_values: {len(all_unique_values)}")

unique_values: 1035


In [199]:
normalize_dict = {}
for value in tqdm(all_unique_values):
    cleaned = normalize(value)
    corrected = correct_spelling(cleaned)
    normalize_dict[value] = corrected

100%|██████████| 1035/1035 [01:27<00:00, 11.83it/s]


In [200]:
def process_cell(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return ""
    
    terms = [t.strip() for t in str(cell).split('##') if t.strip()]
    processed_terms = [normalize_dict.get(t) for t in terms]
    return " ## ".join(processed_terms)

In [201]:
sef['side_processed'] = sef['side'].apply(process_cell)

In [202]:
sef.head()

,id,SideEffect,SideEffects,side,side_processed
0,1,Most side effects do not require any medical a...,Most side effects##Common side effects##Agitat...,Agitation##Extrapyramidal symptoms##Insomnia (...,agitation ## extrapyramidal symptoms ## insomn...
1,2,Most side effects do not require any medical a...,Most side effects##Common side effects##Rectal...,Rectal bleeding##Taste change##Headache##Noseb...,rectal bleeding ## taste change ## headache ##...
2,3,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...,abdominal pain ## high blood pressure ## rash ...
3,4,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...,abdominal pain ## high blood pressure ## rash ...
4,5,Most side effects do not require any medical a...,Most side effects##Common side effects##Abdomi...,Abdominal pain##High blood pressure##Rash##Inj...,abdominal pain ## high blood pressure ## rash ...


In [ ]:
with open('..\\data\\processed\\side_effects_dict.json', 'r', encoding='utf-8') as f:
    mapping_dict = json.load(f)

In [205]:
final_mapping = {}
for original, normalized in normalize_dict.items():
    if normalized in mapping_dict:
        final_mapping[normalized] = mapping_dict[normalized]
    else:
        final_mapping[normalized] = normalized

print(f"Final Dict: {len(final_mapping)} ")

Final Dict: 1031 


In [206]:
def map_cell(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return ""
    
    terms = [t.strip() for t in str(cell).split('##') if t.strip()]
    mapped_terms = [final_mapping.get(t, t) for t in terms]
    unique_mapped = list(dict.fromkeys(mapped_terms))
    return " ## ".join(unique_mapped)

In [207]:
sef['side_mapped'] = sef['side_processed'].apply(map_cell)

In [208]:
sideeffects = sef[["id","side_mapped"]]

In [ ]:
sideeffects.to_csv("..\\data\\processed\\sideeffects.csv",index=False)